In [5]:
import numpy as np
import math

L, d_k, d_v =4,8,8
q = np.random.randn(L, d_k)
k = np.random.randn(L, d_k)
v = np.random.randn(L, d_v)

In [6]:
print("Q\n", q)
print("K\n", k)
print("V\n", v)


Q
 [[ 1.07510353  0.81689291 -1.44394688 -0.13808261  1.42594915 -0.04901616
  -0.95084484  0.43338298]
 [ 0.7617591  -0.04683004 -0.82965953  0.94390529 -0.72217492 -0.18961059
  -0.88746055  0.10463817]
 [-1.53681708 -0.19619631  0.62024274  0.2779034   0.83611567  1.96628781
  -0.57069301  0.30329759]
 [-0.59811228 -0.90122065  0.50363295 -0.08586141  1.58501334  0.11359406
  -1.08358671 -1.87293101]]
K
 [[-0.78456707 -1.57961536 -1.29696137  0.31601943  0.08689382  0.48222319
   0.76791474  0.57212247]
 [ 0.18874371 -2.70027314 -0.18619147 -0.39368977  0.3040868  -1.55595741
  -0.25876482  1.61070853]
 [-1.00060311 -0.01085851 -0.51951429 -0.54732006  0.63134528  0.38641651
  -1.06723736 -0.49304103]
 [ 1.24606496  3.0310354  -0.13578041 -1.74854855  0.40914002  1.27953782
   0.57667671 -0.18282712]]
V
 [[ 0.93149129 -0.90797802  1.12755343  1.49034217  0.43106028  0.31811815
  -1.97554697  1.02661667]
 [-0.08064865 -0.95754543 -1.57542753 -0.44651822 -0.41380774  0.09660726
   0.2


$$
\text{self attention} = softmax\left(\frac{Q \cdot K^T}{\sqrt{d_k}} + M\right)V
$$




In [7]:
#nhân ma trận Q . KT
np.matmul(q, k.T)

array([[-0.68671107, -0.22572442,  1.42353141,  4.14631462],
       [ 0.07483575,  0.52670811, -0.48097858, -1.799548  ],
       [ 1.55516544, -2.15418161,  2.81275778, -0.60632465],
       [-0.49862775, -0.17043512,  3.51806944, -2.88378546]])

In [8]:
#tại sao lại cần phải chia cho sqrt(d_k) khi tính attention score?
q.var(), k.var(),np.matmul(q, k.T).var()

(np.float64(0.8453934010637485),
 np.float64(1.1881754320597122),
 np.float64(3.6567559825930553))

In [9]:
scaled = np.matmul(q, k.T) / math.sqrt(d_k)
q.var(), k.var(), scaled.var()

(np.float64(0.8453934010637485),
 np.float64(1.1881754320597122),
 np.float64(0.4570944978241318))

In [10]:
scaled

array([[-0.24278903, -0.07980564,  0.50329436,  1.46594359],
       [ 0.02645843,  0.18621944, -0.17005161, -0.6362363 ],
       [ 0.54983402, -0.76161821,  0.99446005, -0.21436814],
       [-0.17629153, -0.06025792,  1.24382538, -1.01957213]])

Masking

In [11]:
mask = np.tril(np.ones((L, L)))
mask

array([[1., 0., 0., 0.],
       [1., 1., 0., 0.],
       [1., 1., 1., 0.],
       [1., 1., 1., 1.]])

In [12]:
mask[mask == 0] = -np.inf
mask[mask == 1] = 0

In [13]:
mask

array([[  0., -inf, -inf, -inf],
       [  0.,   0., -inf, -inf],
       [  0.,   0.,   0., -inf],
       [  0.,   0.,   0.,   0.]])

In [14]:
scaled + mask

array([[-0.24278903,        -inf,        -inf,        -inf],
       [ 0.02645843,  0.18621944,        -inf,        -inf],
       [ 0.54983402, -0.76161821,  0.99446005,        -inf],
       [-0.17629153, -0.06025792,  1.24382538, -1.01957213]])

$$
\text{softmax} = \frac{e^{x_i}}{\sum_j e^{x_j}}
$$

In [15]:
def softmax(x):
    return (np.exp(x).T / np.sum(np.exp(x), axis=1)).T

In [16]:
attention = softmax(scaled + mask)
attention

array([[1.        , 0.        , 0.        , 0.        ],
       [0.46014448, 0.53985552, 0.        , 0.        ],
       [0.35343991, 0.09522679, 0.5513333 , 0.        ],
       [0.14945596, 0.16784407, 0.61838959, 0.06431039]])

In [26]:
def softmax(x):
    return(np.exp(x).T/np.sum(np.exp(x), axis = 1)).T

def scaled_dot_Product_attention(q, k, v, mask = None) :
    d_k = q.shape[-1]
    scaled = np.matmul(q, k.T) / math.sqrt(d_k)
    if mask is not None:
        scaled = scaled + mask
    attention = softmax(scaled)
    out = np.matmul(attention, v)
    return out, attention

In [28]:
values,attention = scaled_dot_Product_attention(q, k, v, mask = mask)
print("Q\n", q)
print("K\n", k)
print("V\n", v)
print("Values\n", values)
print("Attention\n", attention)

Q
 [[ 1.07510353  0.81689291 -1.44394688 -0.13808261  1.42594915 -0.04901616
  -0.95084484  0.43338298]
 [ 0.7617591  -0.04683004 -0.82965953  0.94390529 -0.72217492 -0.18961059
  -0.88746055  0.10463817]
 [-1.53681708 -0.19619631  0.62024274  0.2779034   0.83611567  1.96628781
  -0.57069301  0.30329759]
 [-0.59811228 -0.90122065  0.50363295 -0.08586141  1.58501334  0.11359406
  -1.08358671 -1.87293101]]
K
 [[-0.78456707 -1.57961536 -1.29696137  0.31601943  0.08689382  0.48222319
   0.76791474  0.57212247]
 [ 0.18874371 -2.70027314 -0.18619147 -0.39368977  0.3040868  -1.55595741
  -0.25876482  1.61070853]
 [-1.00060311 -0.01085851 -0.51951429 -0.54732006  0.63134528  0.38641651
  -1.06723736 -0.49304103]
 [ 1.24606496  3.0310354  -0.13578041 -1.74854855  0.40914002  1.27953782
   0.57667671 -0.18282712]]
V
 [[ 0.93149129 -0.90797802  1.12755343  1.49034217  0.43106028  0.31811815
  -1.97554697  1.02661667]
 [-0.08064865 -0.95754543 -1.57542753 -0.44651822 -0.41380774  0.09660726
   0.2